# Audio Source Separation - Visualization

This notebook demonstrates audio source separation and visualization of results.

## Setup
Run the following to install dependencies:

In [ ]:
# Install required packages
# !pip install -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '.')

## Load and Analyze Audio

In [ ]:
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SAMPLE_AUDIO = "data/test/sample.wav"  # Replace with your audio file

In [ ]:
def load_audio(path):
    """Load audio file."""
    waveform, sr = torchaudio.load(path)
    return waveform, sr

def plot_waveform(waveform, sr, title="Waveform"):
    """Plot waveform."""
    plt.figure(figsize=(12, 4))
    
    if waveform.shape[0] > 1:
        for i in range(waveform.shape[0]):
            plt.plot(waveform[i].numpy(), label=f"Channel {i+1}")
    else:
        plt.plot(waveform[0].numpy())
    
    plt.title(title)
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_spectrogram(waveform, sr, title="Spectrogram", n_fft=1024):
    """Plot spectrogram using STFT."""
    spec = torchaudio.functional.spectrogram(
        waveform, n_fft=n_fft
    )
    
    plt.figure(figsize=(12, 6))
    plt.imshow(
        torch.log10(spec[0] + 1e-10).numpy(),
        origin="lower",
        aspect="auto",
        cmap="viridis"
    )
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Frequency (Hz)")
    plt.colorbar(label="log magnitude")
    plt.show()

## Analyze Single Track

In [ ]:
# Load and analyze sample audio
if Path(SAMPLE_AUDIO).exists():
    waveform, sr = load_audio(SAMPLE_AUDIO)
    duration = waveform.shape[1] / sr
    
    print(f"Sample Rate: {sr} Hz")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Channels: {waveform.shape[0]}")
    print(f"Samples: {waveform.shape[1]}")
    
    plot_waveform(waveform, sr, "Original Audio Waveform")
    plot_spectrogram(waveform, sr, "Original Audio Spectrogram")
else:
    print(f"Sample file not found: {SAMPLE_AUDIO}")
    print("Please add a test audio file to data/test/")

## Run Source Separation

In [ ]:
from app.separator import SourceSeparator

# Initialize separator
separator = SourceSeparator(
    model_name="htdemucs_ft",
    device="cpu",
    output_dir="output"
)

print("Separator initialized")

In [ ]:
# Run separation
if Path(SAMPLE_AUDIO).exists():
    stems = separator.separate(SAMPLE_AUDIO)
    
    print("\nSeparated stems:")
    for name, path in stems.items():
        print(f"  {name}: {path}")
else:
    print("No sample audio to process")

## Visualize Separated Stems

In [ ]:
# Visualize each stem
if Path(SAMPLE_AUDIO).exists():
    stems_dir = Path("output") / Path(SAMPLE_AUDIO).stem
    
    stem_names = ["drums", "bass", "other", "vocals"]
    
    for stem_name in stem_names:
        stem_path = stems_dir / f"{stem_name}.wav"
        
        if stem_path.exists():
            stem_wave, stem_sr = load_audio(str(stem_path))
            
            plot_waveform(
                stem_wave, 
                stem_sr, 
                f"{stem_name.upper()} - Waveform"
            )
            plot_spectrogram(
                stem_wave, 
                stem_sr, 
                f"{stem_name.upper()} - Spectrogram"
            )

## Audio Playback (optional)

In [ ]:
# Play audio (requires IPyWidgets)
# from IPython.display import Audio, display
# 
# display(Audio(str(stem_path)))

## Calculate Metrics

In [ ]:
from app.metrics import SeparationMetrics

calculator = SeparationMetrics()
print("Metrics calculator initialized")

In [ ]:
# Calculate metrics if reference is available
# results = calculator.evaluate_full_track(
#     reference_dir=Path("data/test/track_name"),
#     estimated_dir=Path("output/track_name")
# )
# 
# print("\n=== Metrics ===")
# for source, metrics in results.items():
#     print(f"\n{source.upper()}:")
#     for metric, value in metrics.items():
#         print(f"  {metric}: {value:.2f} dB")

## Summary

This notebook demonstrates:
- Loading and visualizing audio waveforms and spectrograms
- Running source separation with Demucs
- Visualizing separated stems
- Calculating quality metrics

## Notes
- Processing time depends on hardware (CPU/GPU)
- For best quality, use htdemucs_ft model
- For faster processing, use smaller audio clips for testing